# Train one model

Set `TASK` to coder, judge or selfcritique. Run with `SMOKE = True` first (about 5 min), then `SMOKE = False`. Use an L4 GPU (batch 1) or an A100 (batch 4). Output goes to `checkpoints/`.

`train/run_all.py` trains all three in one go.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/sml-project"
DATA_DIR    = f"{PROJECT_DIR}/data/final_python"
LANG_TAG    = "python"            # "all" for the all-languages row of the experiment
TASK        = "coder"             # "coder" | "judge" | "selfcritique"
SMOKE       = True                # True: quick end-to-end check. False: real training.

MODEL = "Qwen/Qwen3.5-0.8B"

# same for all models
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LEARNING_RATE = 2e-4
EPOCHS        = 2
BATCH_SIZE    = 1                 # per step; effective batch = BATCH_SIZE * GRAD_ACCUM = 16
GRAD_ACCUM    = 16                # batch 4 only fits on an A100
MAX_LEN       = 4096
WARMUP_RATIO  = 0.03
EVAL_STEPS    = 200
SEED          = 42

RUN_NAME   = f"{LANG_TAG}_{TASK}" + ("_smoke" if SMOKE else "")
OUTPUT_DIR = f"{PROJECT_DIR}/checkpoints/{RUN_NAME}"
print("Run:", RUN_NAME, "->", OUTPUT_DIR)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Qwen3.5 needs a recent transformers
!pip install -q -U transformers accelerate peft evalplus
# the torchao on Colab breaks peft, we don't use it
!pip uninstall -y -q torchao
# optional, faster linear attention
!pip install -q flash-linear-attention || echo "flash-linear-attention not installed (optional)"

import sys, torch, transformers, peft
sys.path[:0] = [f"{PROJECT_DIR}/{d}" for d in ["data", "train", "eval"]]
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > L4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

## Data

In [ ]:
from collections import Counter
from sft_utils import load_jsonl
import sft_utils as S

train_rows = load_jsonl(f"{DATA_DIR}/{TASK}_train.jsonl", limit=200 if SMOKE else None)
val_rows   = load_jsonl(f"{DATA_DIR}/{TASK}_val.jsonl",   limit=50 if SMOKE else None)
print(f"train {len(train_rows)}  val {len(val_rows)}")
print("kinds:", dict(Counter(r["kind"] for r in train_rows)))
thinking_runs = {S.uses_thinking(r) for r in train_rows}
print("thinking mode in training:", thinking_runs)

## Model and LoRA

In [ ]:
from transformers import AutoTokenizer

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL)
import transformers as tf
# Qwen3.5-0.8B is multimodal, try the classes that can load it
model = None
for cls_name in ["AutoModelForMultimodalLM", "AutoModelForImageTextToText", "AutoModelForCausalLM"]:
    cls = getattr(tf, cls_name, None)
    if cls is None:
        continue
    try:
        model = cls.from_pretrained(MODEL, torch_dtype=dtype, device_map="cuda")
        print("loaded with", cls_name)
        break
    except (ValueError, KeyError) as err:
        print(cls_name, "failed:", str(err)[:120])
assert model is not None, "could not load the model: upgrade transformers (pip install -U transformers)"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")

In [ ]:
# check that enable_thinking changes the prompt
probe = [{"role": "user", "content": "hi"}]
off = tokenizer.apply_chat_template(probe, tokenize=False, add_generation_prompt=True, enable_thinking=False)
on = tokenizer.apply_chat_template(probe, tokenize=False, add_generation_prompt=True, enable_thinking=True)
print("thinking OFF ends with:", repr(off[-60:]))
print("thinking ON  ends with:", repr(on[-60:]))
assert off != on, ("enable_thinking has no effect on this chat template. Stop and check the "
                   "model card for how thinking is switched on/off before continuing.")

In [ ]:
from peft import LoraConfig, get_peft_model
import sft_utils as S

targets = S.find_lora_targets(model)
print("LoRA target layers:", targets)
assert targets, "no linear layers found; print(model) and set target_modules by hand"

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=targets, task_type="CAUSAL_LM"))
model.print_trainable_parameters()

## Tokenize and check the loss masking

The loss should only cover the answer, not the prompt.

In [ ]:
import statistics
import sft_utils as S

train_ds, dropped_train = S.build_dataset(tokenizer, train_rows, MAX_LEN)
val_ds, dropped_val = S.build_dataset(tokenizer, val_rows, MAX_LEN)
print(f"dropped as longer than {MAX_LEN} tokens: train {dropped_train}, val {dropped_val}")
lengths = [len(f["input_ids"]) for f in train_ds]
print(f"tokens per example: median {statistics.median(lengths):.0f}, max {max(lengths)}")

for kind in sorted({r["kind"] for r in train_rows}):
    i = next(i for i, r in enumerate(train_rows) if r["kind"] == kind)
    feats = S.build_features(tokenizer, train_rows[i], MAX_LEN)
    print(f"\n--- {kind} ---\n" + S.describe_masking(tokenizer, feats))
    full = tokenizer.decode(feats["input_ids"])
    assert full.count("<think>") <= 1, "think block appears twice: chat template mismatch"
    prompt_ids = [t for t, l in zip(feats["input_ids"], feats["labels"]) if l == S.IGNORE_INDEX]
    inference_prompt = S.render_prompt(tokenizer, train_rows[i]["messages"][:-1], S.uses_thinking(train_rows[i]))
    assert tokenizer.decode(prompt_ids) == tokenizer.decode(tokenizer(inference_prompt, add_special_tokens=False)["input_ids"]), \
        "training prompt differs from inference prompt"
print("\nMasking checks passed")

## Train

In [ ]:
from transformers import Trainer, TrainingArguments
import inspect
import sft_utils as S

use_bf16 = torch.cuda.is_bf16_supported()
steps = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS
total_steps = 30 if SMOKE else steps
warmup_steps = max(1, int(WARMUP_RATIO * total_steps))     # 3% of training, as a step count

settings = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=30 if SMOKE else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    bf16=use_bf16, fp16=not use_bf16,
    logging_steps=5 if SMOKE else 10,
    eval_strategy="steps", eval_steps=10 if SMOKE else EVAL_STEPS,
    save_strategy="steps", save_steps=10 if SMOKE else EVAL_STEPS, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    group_by_length=True,
    remove_unused_columns=False,
    report_to="none",
    prediction_loss_only=True,      # validation needs only the loss, not the 248k-wide logits
    seed=SEED, data_seed=SEED,
)
# drop arguments this transformers version doesn't have
known = set(inspect.signature(TrainingArguments.__init__).parameters)
dropped = sorted(k for k in settings if k not in known)
if dropped:
    print("Not supported by this transformers version, skipped:", dropped)
args = TrainingArguments(**{k: v for k, v in settings.items() if k in known})

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=S.PadCollator(tokenizer.pad_token_id))
print(f"~{steps} optimizer steps planned" + (" (SMOKE: capped at 30)" if SMOKE else "")
      + f", warmup {warmup_steps} steps")
trainer.train()

In [ ]:
import json, matplotlib.pyplot as plt

history = trainer.state.log_history
json.dump(history, open(f"{OUTPUT_DIR}/log_history.json", "w"), indent=1)
tr = [(h["step"], h["loss"]) for h in history if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
plt.plot(*zip(*tr), label="train loss"); plt.plot(*zip(*ev), "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title(RUN_NAME); plt.show()
best = min(ev, key=lambda x: x[1])
print(f"best validation loss {best[1]:.3f} at step {best[0]} (this checkpoint is kept)")

If validation loss goes up while train loss goes down, it is overfitting. The best checkpoint is kept.

## Sample outputs

In [ ]:
import eval_utils as E
import prompts as P

model.eval()
samples = val_rows[:3]
convs = [r["messages"][:-1] for r in samples]
gens = E.generate(model, tokenizer, convs, enable_thinking=(TASK == "selfcritique"),
                  max_new_tokens=1500, batch_size=3)
for r, g in zip(samples, gens):
    print(f"=== {r['task_id']} ({r['kind']}) ===")
    print(g["text"][:1500])
    if TASK == "judge":
        print("parsed:", P.parse_judge_output(g["text"]), "| dataset rating:", r["rating"])
    print()

## Merge and save

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained(f"{OUTPUT_DIR}/merged")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged")
json.dump({"task": TASK, "lang": LANG_TAG, "smoke": SMOKE, "base_model": MODEL,
           "lora": [LORA_R, LORA_ALPHA, LORA_DROPOUT], "lr": LEARNING_RATE, "epochs": EPOCHS,
           "effective_batch": BATCH_SIZE * GRAD_ACCUM, "max_len": MAX_LEN, "seed": SEED,
           "train_examples": len(train_ds), "dropped_train": dropped_train,
           "best_eval_loss": best[1], "best_step": best[0]},
          open(f"{OUTPUT_DIR}/merged/run_config.json", "w"), indent=1)
print("Saved", f"{OUTPUT_DIR}/merged")
print("Next: open 01_evaluate, set MODEL to this folder and RUN_NAME =", repr(RUN_NAME))